# Prepping Data Challenge - Week 05 

We want to answer the following **questions:**
- What are the details of the customers who have booked flights and which routes are they travelling on?
- Which customers are yet to book a flight in 2024?
- Which flights are yet to be booked by customers in 2024?

## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Import Data

In [2]:
flights_data = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_5_Prep_Air_2024_Flights.csv')
ticket_sales_data = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_5_Prep_Air_Ticket_Sales.csv')
customers_data = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_5_Prep_Air_Customers.csv')

## Data Exploration and Cleaning

In [3]:
flights_data.head()

,Date,Flight Number,From,To
0,2024-11-22,PA001,London,New York
1,2024-11-23,PA001,London,New York
2,2024-11-23,PA002,New York,London
3,2024-11-24,PA001,London,New York
4,2024-11-27,PA001,London,New York


In [4]:
ticket_sales_data.head()

,Date,Flight Number,Customer ID,Ticket Price
0,2024-01-03,PA001,232,818.99
1,2024-01-03,PA001,293,1947.99
2,2024-01-03,PA001,472,1350.99
3,2024-01-03,PA001,572,905.99
4,2024-01-03,PA001,1191,567.99


In [5]:
customers_data.head()

,Customer ID,Last Date Flown,first_name,last_name,email,gender
0,1,2023-01-05,Denyse,Gebuhr,dgebuhr0@vinaora.com,Female
1,2,2023-10-05,Keene,Devennie,kdevennie1@plala.or.jp,Male
2,3,2023-11-09,Tyler,McGrail,tmcgrail2@nyu.edu,Male
3,4,2023-11-22,Drusi,Ibeson,dibeson3@hostgator.com,Female
4,5,2023-12-23,Stanwood,Seacroft,sseacroft4@wikispaces.com,Male


All of these Datasets contain a `Date` column. This column needs to be changed to datetime format

In [6]:
flights_data['Date'] = pd.to_datetime(flights_data['Date'])
ticket_sales_data['Date'] = pd.to_datetime(ticket_sales_data['Date'])
customers_data['Last Date Flown'] = pd.to_datetime(customers_data['Last Date Flown'])

## Challenges

I want to try to answer the questions without looking at the Step by Step guide first

### 1. What are the details of the customers who have booked flights, and which route are they traveling on?

This just sounds like a large inner join across the board. ~The one thing that is weird is that there are multiple flight numbers on the same day sometimes, and there does not seem to be a way for me to get identify which customer was on which flight~ Nevermind, I can match via customer ID

In [7]:
customers_data.nunique()

Customer ID        9999
Last Date Flown     394
first_name         5773
last_name          9006
email              9999
gender                8
dtype: int64

**Important:** Customer ID is unique (Email is unique as well)

#### Join `customers_data` with `ticket_sales_data`

In [8]:
last_flight_by_customer_data = pd.merge(customers_data, ticket_sales_data, on=['Customer ID'])

In [9]:
last_flight_by_customer_data.shape

(44768, 9)

#### Join `last_flight_by_customer_data` with `flights_data`

In [10]:
customer_details_data = pd.merge(last_flight_by_customer_data, flights_data, on=['Date', 'Flight Number'])

I'm not sure, if we are supposed to keep `ticket price`. I will do so for now.

In [46]:
customer_details_data.head()

,Customer ID,Last Date Flown,first_name,last_name,email,gender,Date,Flight Number,Ticket Price,From,To
0,1,2023-01-05,Denyse,Gebuhr,dgebuhr0@vinaora.com,Female,2024-06-03,PA003,1430.99,London,Perth
1,1,2023-01-05,Denyse,Gebuhr,dgebuhr0@vinaora.com,Female,2024-05-12,PA005,776.99,London,Tokyo
2,1,2023-01-05,Denyse,Gebuhr,dgebuhr0@vinaora.com,Female,2024-05-12,PA006,708.99,Tokyo,London
3,1,2023-01-05,Denyse,Gebuhr,dgebuhr0@vinaora.com,Female,2024-05-17,PA005,1027.99,London,Tokyo
4,1,2023-01-05,Denyse,Gebuhr,dgebuhr0@vinaora.com,Female,2024-05-17,PA006,1821.99,Tokyo,London


In [47]:
customer_details_data.shape

(44768, 11)

### 2. Which customers are yet to book a flight in 2024?

In [15]:
current_date = np.datetime64('2024-01-31')

As `customer_details_data` contains all customers which have booked a flight in 2024, we can simply return a list of `Customer ID`s. We can then check with the `customers_data` to see which IDs are missing

In [20]:
customers_that_have_booked = customer_details_data['Customer ID'].unique()

In [22]:
customers_that_have_booked.shape

(8739,)

In [25]:
customers_not_booked = customers_data[~customers_data['Customer ID'].isin(customers_that_have_booked)]

In [31]:
customers_not_booked['Days since Last Flight'] = (current_date - customers_not_booked['Last Date Flown'].copy()).dt.days

C:\Users\tobia\AppData\Local\Temp\ipykernel_4436\862228627.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customers_not_booked['Days since Last Flight'] = (current_date - customers_not_booked['Last Date Flown'].copy()).dt.days


In [42]:
def create_customer_categories(last_date):
    months = int((current_date - last_date) / pd.Timedelta(days=30.44))
    if months < 3:
        return 'Recent fliers (flown within the last 3 months)'
    elif months < 6:
        return 'Taking a break (3-6 months since last flight)'
    elif months < 9:
        return 'Been away a while (6-9 months since last flight)'
    else:
        return 'Lapsed Customer (over 9 months since last flight)'

In [44]:
customers_not_booked['Customer Category'] = customers_not_booked['Last Date Flown'].apply(create_customer_categories)

C:\Users\tobia\AppData\Local\Temp\ipykernel_4436\1640140920.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customers_not_booked['Customer Category'] = customers_not_booked['Last Date Flown'].apply(create_customer_categories)


In [48]:
customers_not_booked.head()

,Customer ID,Last Date Flown,first_name,last_name,email,gender,Days since Last Flight,Customer Category
2,3,2023-11-09,Tyler,McGrail,tmcgrail2@nyu.edu,Male,83,Recent fliers (flown within the last 3 months)
8,9,2023-06-05,Binnie,Jeckell,bjeckell8@123-reg.co.uk,Female,240,Been away a while (6-9 months since last flight)
13,14,2023-05-02,Ashil,Tetlow,atetlowd@woothemes.com,Female,274,Lapsed Customer (over 9 months since last flight)
14,15,2023-05-18,Ayn,Bengtson,abengtsone@bloomberg.com,Female,258,Been away a while (6-9 months since last flight)
23,24,2023-02-15,Grace,Piesing,gpiesingn@zdnet.com,Female,350,Lapsed Customer (over 9 months since last flight)


In [49]:
customers_not_booked.shape

(1260, 8)

### 3. Which flights are yet to be booked in 2024?

In [61]:
booked_flights = customer_details_data['Flight Number'] + ' ' + customer_details_data['Date'].astype(str)

In [62]:
unbooked_flights = flights_data[~(flights_data['Flight Number'] + ' ' + flights_data['Date'].astype(str)).isin(booked_flights.unique())]

In [64]:
unbooked_flights['Flight unbooked as of'] = current_date

C:\Users\tobia\AppData\Local\Temp\ipykernel_4436\4247597800.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unbooked_flights['Flight unbooked as of'] = current_date


In [67]:
unbooked_flights.head()

,Date,Flight Number,From,To,Flight unbooked as of
0,2024-11-22,PA001,London,New York,2024-01-31
1,2024-11-23,PA001,London,New York,2024-01-31
2,2024-11-23,PA002,New York,London,2024-01-31
3,2024-11-24,PA001,London,New York,2024-01-31
4,2024-11-27,PA001,London,New York,2024-01-31


In [68]:
unbooked_flights.shape

(296, 5)

## Save results

In [66]:
customer_details_data.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/05_customer_details.csv')
customers_not_booked.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/05_customers_not_booked.csv')
unbooked_flights.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/05_flights_not_booked.csv')